In [ ]:
# standardowe biblioteki
import os, sys, time, glob, random, base64, shutil, pathlib, io, json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

# Colab wyświetlanie
from IPython.display import Javascript, display, Audio, Image as DImage
from google.colab.output import eval_js

# TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mnv2_preprocess
from tensorflow.keras.models import load_model
print("TF:", tf.__version__)



In [ ]:
BASE_DIR='/content/projekt_kpn'
RAW_DIR= BASE_DIR + '/raw'
DATA_DIR= BASE_DIR + '/data'

KATEGORIE= ['Kamień', 'Papier', 'Nożyce']
IMG_SIZE= (224, 224)
TRAIN_RATIO= 0.8

In [ ]:
for k in KATEGORIE:
  os.makedirs(os.path.join(RAW_DIR, k), exist_ok= True)

In [ ]:
def zbierz_proby(kategoria, ile_min=20, jakosc=0.9, lustrzane=True):
    # proste GUI w JS: przycisk "Zrób zdjęcie" i "Zakończ"
    target = os.path.join(RAW_DIR, kategoria)
    os.makedirs(target, exist_ok=True)

    js = Javascript('''
      async function sesja(nMin, q) {
        const div = document.createElement('div');
        const btn = document.createElement('button');
        btn.textContent = 'Zrób zdjęcie';
        btn.style.fontSize = '16px';
        btn.style.marginRight = '8px';
        const done = document.createElement('button');
        done.textContent = 'Zakończ';
        done.disabled = true;
        const info = document.createElement('span');
        info.style.marginLeft = '10px';
        info.textContent = '0 / ' + nMin;

        div.appendChild(btn);
        div.appendChild(done);
        div.appendChild(info);

        const video = document.createElement('video');
        video.style.display = 'block';
        video.style.maxWidth = '480px';
        const stream = await navigator.mediaDevices.getUserMedia({video:true});
        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

        const images = [];
        btn.onclick = () => {
          const canvas = document.createElement('canvas');
          canvas.width = video.videoWidth;
          canvas.height = video.videoHeight;
          canvas.getContext('2d').drawImage(video, 0, 0);
          images.push(canvas.toDataURL('image/jpeg', q));
          info.textContent = images.length + ' / ' + nMin;
          if (images.length >= nMin) done.disabled = false;
        };

        await new Promise(resolve => done.onclick = resolve);
        stream.getTracks().forEach(t => t.stop());
        div.remove();
        return images;
      }
    ''')
    display(js)
    data_urls = eval_js(f"sesja({ile_min}, {jakosc})")


    zapisane = 0
    for i, d in enumerate(data_urls):
        raw = base64.b64decode(d.split(',')[1])
        fname = os.path.join(target, f"{kategoria}_{int(time.time()*1000)}_{i:03d}.jpg")
        with open(fname, 'wb') as f:
            f.write(raw)
        zapisane += 1

        if lustrzane:
            img = Image.open(io.BytesIO(raw)).convert('RGB')
            img = ImageOps.mirror(img)
            fname2 = os.path.join(target, f"{kategoria}_{int(time.time()*1000)}_{i:03d}_m.jpg")
            img.save(fname2, "JPEG", quality=int(jakosc*100))
            zapisane += 1

    print("Zapisano:", zapisane, "plików do", target)


In [ ]:
zbierz_proby('Papier', ile_min=50)

In [ ]:
zbierz_proby('Kamień', ile_min=50)

In [ ]:
zbierz_proby('Nożyce', ile_min=50)

In [ ]:
def podziel_train_test():
    if os.path.exists(DATA_DIR):
        shutil.rmtree(DATA_DIR)
    for split in ["train", "test"]:
        for k in KATEGORIE:
            os.makedirs(os.path.join(DATA_DIR, split, k), exist_ok=True)

    for k in KATEGORIE:
        pliki = glob.glob(os.path.join(RAW_DIR, k, "*"))
        pliki = [p for p in pliki if os.path.isfile(p)]
        random.shuffle(pliki)
        n = len(pliki)
        n_train = int(n * TRAIN_RATIO)
        train_files = pliki[:n_train]
        test_files  = pliki[n_train:]
        for src in train_files:
            shutil.copy2(src, os.path.join(DATA_DIR, "train", k, os.path.basename(src)))
        for src in test_files:
            shutil.copy2(src, os.path.join(DATA_DIR, "test", k, os.path.basename(src)))
    print("Dane w", DATA_DIR)

In [ ]:
podziel_train_test()

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 32

def wczytaj_dane():
  train_ds= image_dataset_from_directory(
      os.path.join(DATA_DIR, 'train'),
      image_size=IMG_SIZE,
      batch_size=BATCH_SIZE,
      label_mode='categorical',
      shuffle=True
  )
  test_ds= image_dataset_from_directory(
      os.path.join(DATA_DIR, 'test'),
      image_size=IMG_SIZE,
      batch_size=BATCH_SIZE,
      label_mode='categorical',
      shuffle=False
  )

  return train_ds.prefetch(AUTOTUNE), test_ds.prefetch(AUTOTUNE), train_ds.class_names

In [ ]:
def zbuduj_model(n_klas):
  wej= layers.Input( shape = IMG_SIZE + (3,))
  x = layers.RandomFlip('horizontal')(wej)
  x= layers.RandomRotation(0.05)(x)
  x= layers.RandomZoom(0.1)(x)
  x= mnv2_preprocess(x)

  baza= MobileNetV2(include_top=False, input_tensor=x, weights='imagenet', pooling='avg')
  baza.trainable= False

  x= layers.Dropout(0.2)(baza.output)
  wyj= layers.Dense(n_klas, activation='softmax')(x)
  model= models.Model(wej, wyj)
  model.compile(
      optimizer='adam',
      loss='categorical_crossentropy',
      metrics=['accuracy']
  )

  return model


In [ ]:
train_ds, test_ds, class_names= wczytaj_dane()


model= zbuduj_model(len(class_names))
hist= model.fit(train_ds, validation_data = test_ds, epochs= 6)
loss, acc= model.evaluate(test_ds)

print('Test acc: acc')

plt.plot(hist.history['accuracy'], label='train')
plt.plot(hist.history['val_accuracy'], label='val')
plt.legend()
plt.title('dokładnośc')
plt.show()

In [ ]:
def take_photo(nazwa="foto.jpg", jakosc=0.9):
    js = Javascript('''
      async function f(q) {
        const div = document.createElement('div');
        const b = document.createElement('button');
        b.textContent = 'Zrób zdjęcie';
        b.style.fontSize = '16px';
        b.style.marginBottom = '8px';
        div.appendChild(b);

        const video = document.createElement('video');
        video.style.display='block';
        video.style.maxWidth='480px';
        const stream = await navigator.mediaDevices.getUserMedia({video:true});
        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

        await new Promise(resolve => b.onclick = resolve);
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getTracks().forEach(t => t.stop());
        div.remove();
        return canvas.toDataURL('image/jpeg', q);
      }
    ''')
    display(js)
    data = eval_js(f"f({jakosc})")
    raw = base64.b64decode(data.split(',')[1])
    with open(nazwa, 'wb') as f:
        f.write(raw)
    return nazwa

In [ ]:
def przewidz(path, mirror=False, topk=3):
    x = przygotowanie(path, mirror=mirror)
    p = model.predict(x, verbose=0)[0]
    idx = int(np.argmax(p))
    etyk = class_names[idx]              # surowa etykieta (nazwa folderu)
    tytul = etyk.replace("_"," ").title()
    pew = float(p[idx])

    print(f"Przewidziano: {tytul} ({etyk}), pewność: {pew:.2f}")
    top = np.argsort(p)[-topk:][::-1]
    print("Top-3:")
    for i in top:
        print(f"  {class_names[i]}: {p[i]:.3f}")

In [ ]:
def przygotowanie(path, rozmiar=IMG_SIZE, mirror=False):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    m = min(w, h)
    img = img.crop(((w-m)//2, (h-m)//2, (w+m)//2, (h+m)//2)).resize(rozmiar)
    if mirror:
        img = ImageOps.mirror(img)
    x = np.asarray(img).astype("float32")
    x = np.expand_dims(x, 0)  # (1,H,W,3) – preprocess jest w MODEL-U
    return x

In [ ]:
foto= take_photo('foto.jpg')
przewidz(foto, mirror=False)

In [ ]:
def kto_wygrywa(gracz, komp):
  if gracz== komp: return 'remis'
  if (gracz == 'Kamień' and komp== 'Nożyce') or \
  (gracz== 'Nożyce' and komp=='Papier') or \
  (gracz== 'Kamień' and komp== 'Nożyce'):
    return 'gracz'
  return 'komputer'


In [ ]:
def przewidz_ruch_zdjecia(path, mirror=False):
  x= przygotowanie(path, mirror=mirror)
  p= model.predict(x, verbose=0)[0]
  idx=int(np.argmax(p))
  etyk= class_names[idx]
  return etyk

In [ ]:
RUCHY=['Papier', 'Kamień', 'Nożyce']
WYNIK={'gracz':0, 'komputer': 0, 'remis': 0}
for runda in range(1,4):
  print(f'\n=== RUNDA {runda} / 3 ===')
  path = take_photo(f'gra{runda}.jpg', jakosc=0.9 )

  gracz= przewidz_ruch_zdjecia(path, mirror=False)

  komp= random.choice(RUCHY)
  wynik= kto_wygrywa(gracz, komp)
  WYNIK[wynik]+= 1

  print('===== PAPIER * KAMIEŃ * NOŻYCE =====')
  print(f'Twój ruch:      {gracz}')
  print(f'Ruch komputera: {komp}')
  if wynik== 'remis':
    print(f'Wynik rundy: REMIS')
  else:
    print(f'Wynik rundy: WYGRYWA {wynik.upper}')

  print(f'Tablica Wyników: runda{runda} wynik gracza{WYNIK['gracz']} wynik komputera: {WYNIK['komputer']} remisy: {WYNIK['remis']}')


=== RUNDA 1 / 3 ===


<IPython.core.display.Javascript object>

AttributeError: 'NoneType' object has no attribute 'split'

In [ ]:
print('\n=== PODSUMOWANIE MECZU ===')
print(f'Tablica Wyników: runda{runda} wynik gracza{WYNIK['gracz']} wynik komputera: {WYNIK['komputer']} remisy: {WYNIK['remis']}')
if WYNIK['gracz'] > WYNIK['komputer']:
  print('Wygryłeś mecz!')
elif WYNIK['gracz'] < WYNIK['komputer']:
  print('Komputer wygrywa mecz!')

else:
  print('Mecz zakończony remisem.')